# 🔮 Predictive CNN: Future Anomaly Forecasting for Bridge Monitoring

**Author**: ComplexBridges ML Team  
**Date**: December 2025  
**Model Type**: 1D Convolutional Neural Network for Predictive Anomaly Detection

---

## 📋 Overview

This notebook implements a **Predictive CNN** that forecasts **future anomalies** with risk scores, providing **early warning** before anomalies occur.

### ⚠️ CORRECTED VERSION
This version fixes time unit calculations and adds support for **3-5 minute prediction horizons**.

### Key Features:
- **Correct time units**: All calculations properly use seconds/minutes
- **Configurable prediction horizons**: Test 5 seconds, 3 minutes, or 5 minutes ahead
- **True predictive analysis**: Evaluate if patterns exist before anomalies
- **Honest metrics reporting**: Median lead times, not just maximums

### Data Overview:
```
Sampling Rate:  10 Hz (10 samples per second)
Total Duration: 30 minutes
Total Samples:  18,000 timesteps per sensor
```

### Prediction Horizon Options:
| Horizon | Timesteps | Use Case |
|---------|-----------|----------|
| 5 seconds | 50 | Immediate automated response |
| 3 minutes | 1,800 | Alert operators, prepare inspection |
| 5 minutes | 3,000 | Schedule maintenance team |

---

## 1. Setup & Imports

In [ ]:
# Install required packages (run once)
!pip install pandas numpy scikit-learn tensorflow matplotlib seaborn joblib -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import os
import json
import joblib
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    precision_recall_curve, roc_curve, f1_score
)

# TensorFlow/Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print("✅ All packages imported successfully!")
print(f"TensorFlow version: {tf.__version__}")
print(f"Pandas version: {pd.__version__}")

## 2. Configuration

### ⚠️ IMPORTANT: Time Unit Clarification

```
Sampling Rate = 10 Hz = 10 samples per SECOND

Conversion formulas:
- timesteps_to_seconds = timesteps / 10
- seconds_to_timesteps = seconds * 10
- minutes_to_timesteps = minutes * 60 * 10 = minutes * 600
```

In [ ]:
# =============================================================================
# CONFIGURATION - EDIT PREDICTION_HORIZON_MINUTES TO TEST DIFFERENT HORIZONS
# =============================================================================

# File paths
DATA_PATH = "../data/ipmb_5sensors_30min_1_to_10hz.csv"
MODEL_DIR = "artifacts/predictive_cnn"

# CRITICAL: Sampling rate (DO NOT CHANGE)
SAMPLE_RATE_HZ = 10  # 10 samples per second

# =============================================================================
# 🎯 PREDICTION HORIZON SELECTION
# =============================================================================
# Choose ONE of the following options:
#   - 'SHORT':  5 seconds  (50 timesteps)  - Immediate automated response
#   - 'MEDIUM': 3 minutes  (1800 timesteps) - Alert operators
#   - 'LONG':   5 minutes  (3000 timesteps) - Schedule maintenance
#   - 'CUSTOM': Set CUSTOM_HORIZON_SECONDS below

PREDICTION_MODE = 'MEDIUM'  # <-- CHANGE THIS TO TEST DIFFERENT HORIZONS
CUSTOM_HORIZON_SECONDS = 120  # Only used if PREDICTION_MODE = 'CUSTOM'

# =============================================================================

# Calculate prediction horizon based on mode
if PREDICTION_MODE == 'SHORT':
    PREDICTION_HORIZON_SECONDS = 5
elif PREDICTION_MODE == 'MEDIUM':
    PREDICTION_HORIZON_SECONDS = 180  # 3 minutes
elif PREDICTION_MODE == 'LONG':
    PREDICTION_HORIZON_SECONDS = 300  # 5 minutes
elif PREDICTION_MODE == 'CUSTOM':
    PREDICTION_HORIZON_SECONDS = CUSTOM_HORIZON_SECONDS
else:
    raise ValueError(f"Invalid PREDICTION_MODE: {PREDICTION_MODE}")

# Convert to timesteps
PREDICTION_HORIZON = PREDICTION_HORIZON_SECONDS * SAMPLE_RATE_HZ

# Historical window (what the model observes)
WINDOW_SIZE_SECONDS = 5  # Observe 5 seconds of history
WINDOW_SIZE = WINDOW_SIZE_SECONDS * SAMPLE_RATE_HZ  # 50 timesteps

# Sliding window stride
STRIDE = 10

# =============================================================================
# ✅ EARLY-WARNING REQUIREMENT ("SLA")
# =============================================================================
# We only count a prediction as useful if it provides at least this much lead time.
# Set to 0 to disable (i.e., any anomaly anywhere in the horizon counts).
MIN_LEAD_SECONDS = 30
MIN_LEAD_TIMESTEPS = MIN_LEAD_SECONDS * SAMPLE_RATE_HZ

# If the required lead time is >= horizon, it's impossible; disable to avoid confusion.
if MIN_LEAD_SECONDS >= PREDICTION_HORIZON_SECONDS:
    print(f"⚠️  MIN_LEAD_SECONDS={MIN_LEAD_SECONDS}s is >= horizon={PREDICTION_HORIZON_SECONDS}s; setting MIN_LEAD_SECONDS=0")
    MIN_LEAD_SECONDS = 0
    MIN_LEAD_TIMESTEPS = 0

# Training parameters
BATCH_SIZE = 64
EPOCHS = 50
LEARNING_RATE = 0.001

# Create output directory
os.makedirs(MODEL_DIR, exist_ok=True)

# Helper functions for time conversion
def timesteps_to_seconds(ts):
    return ts / SAMPLE_RATE_HZ

def timesteps_to_minutes(ts):
    return ts / SAMPLE_RATE_HZ / 60

def format_time(timesteps):
    """Format timesteps as human-readable time"""
    seconds = timesteps / SAMPLE_RATE_HZ
    if seconds < 60:
        return f"{seconds:.1f} seconds"
    else:
        return f"{seconds/60:.1f} minutes ({seconds:.0f} seconds)"

# Display configuration
print("="*70)
print("PREDICTIVE CNN CONFIGURATION (CORRECTED)")
print("="*70)
print(f"📊 Data source: {DATA_PATH}")
print(f"📁 Model directory: {MODEL_DIR}")
print(f"")
print(f"⏱️  Sampling rate: {SAMPLE_RATE_HZ} Hz ({SAMPLE_RATE_HZ} samples/second)")
print(f"")
print(f"👁️  Historical window: {WINDOW_SIZE} timesteps = {format_time(WINDOW_SIZE)}")
print(f"🔮 Prediction horizon: {PREDICTION_HORIZON} timesteps = {format_time(PREDICTION_HORIZON)}")
print(f"⏳ Required minimum lead time: {MIN_LEAD_SECONDS}s ({MIN_LEAD_TIMESTEPS} timesteps)")
print(f"🎯 Mode: {PREDICTION_MODE}")
print(f"")
print(f"🧠 Batch size: {BATCH_SIZE}")
print(f"🔄 Max epochs: {EPOCHS}")
print(f"📈 Learning rate: {LEARNING_RATE}")
print(f"")
print("="*70)
print(f"🎯 GOAL: Predict anomalies {format_time(PREDICTION_HORIZON)} in advance")
print("="*70)

## 3. Data Loading

In [ ]:
print(f"Loading data from {DATA_PATH}...")

df = pd.read_csv(DATA_PATH)

print(f"\n✅ Data loaded successfully!")
print(f"Total rows: {len(df):,}")
print(f"\nDataset shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")

df.head()

In [ ]:
# Data exploration
print("="*70)
print("DATA EXPLORATION")
print("="*70)

print("\n📊 Sensor Type Distribution:")
print(df['sensor_type'].value_counts())

print("\n🔍 Unique Sensors:")
print(df['sensor_id'].unique())

print("\n⚠️  Anomaly Distribution:")
anomaly_counts = df['anomaly'].value_counts()
print(f"Normal (0):  {(df['anomaly'] == 0).sum():,} ({(df['anomaly'] == 0).sum() / len(df) * 100:.2f}%)")
print(f"Anomaly (1): {(df['anomaly'] == 1).sum():,} ({(df['anomaly'] == 1).sum() / len(df) * 100:.2f}%)")

# Visualize anomaly distribution
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Sensor types
df['sensor_type'].value_counts().plot(kind='bar', ax=axes[0], color='skyblue')
axes[0].set_title('Sensor Type Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Sensor Type')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

# Anomaly distribution
anomaly_counts.plot(kind='bar', ax=axes[1], color=['green', 'red'])
axes[1].set_title('Anomaly Distribution', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Label')
axes[1].set_ylabel('Count')
axes[1].set_xticklabels(['Normal', 'Anomaly'], rotation=0)

plt.tight_layout()
plt.show()

## 4. Data Preprocessing

In [ ]:
print("Preprocessing data...")

# Parse timestamps
df['timestamp'] = pd.to_datetime(df['timestamp'], format='ISO8601')
df = df.sort_values('timestamp').reset_index(drop=True)

print(f"✅ Timestamps parsed and data sorted")
print(f"Time range: {df['timestamp'].min()} to {df['timestamp'].max()}")

duration_seconds = (df['timestamp'].max() - df['timestamp'].min()).total_seconds()
print(f"Duration: {duration_seconds:.1f} seconds ({duration_seconds/60:.1f} minutes)")

# Verify sampling rate
n_unique_timestamps = df['timestamp'].nunique()
calculated_rate = n_unique_timestamps / duration_seconds
print(f"\n📊 Data verification:")
print(f"   Unique timestamps: {n_unique_timestamps:,}")
print(f"   Calculated sampling rate: {calculated_rate:.1f} Hz")
print(f"   Expected sampling rate: {SAMPLE_RATE_HZ} Hz")

## 5. Create Sensor Pivot Table

In [ ]:
print("Creating sensor pivot table...")

# Create sensor key
df['sensor_key'] = df['sensor_type'] + '_' + df['sensor_id']

# Pivot sensor values
pivot = df.pivot_table(
    index='timestamp',
    columns='sensor_key',
    values='value',
    aggfunc='first'
)

# Fill missing values
pivot = pivot.ffill().bfill()

# Get anomaly labels (aggregate across sensors)
anomaly_pivot = df.pivot_table(
    index='timestamp',
    columns='sensor_key',
    values='anomaly',
    aggfunc='max'
)
labels = anomaly_pivot.max(axis=1)

print(f"\n✅ Pivot table created")
print(f"Pivot shape: {pivot.shape} (timestamps, sensors)")
print(f"Sensor columns: {list(pivot.columns)}")
print(f"Labels shape: {labels.shape}")

print(f"\nFirst few rows of pivot table:")
pivot.head()

## 6. Normalize Data

In [ ]:
print("Normalizing sensor data...")

normalized = pivot.copy()
scalers = {}

for col in pivot.columns:
    scaler = StandardScaler()
    normalized[col] = scaler.fit_transform(pivot[[col]])
    scalers[col] = scaler

print(f"\n✅ Normalized {len(pivot.columns)} sensor channels")
print(f"\nNormalization stats for each sensor:")
for col in pivot.columns:
    print(f"  {col}: mean={normalized[col].mean():.6f}, std={normalized[col].std():.6f}")

## 7. 🔍 Anomaly Timeline Analysis

Before creating sequences, let's understand **when** anomalies occur to evaluate if prediction is feasible.

In [ ]:
print("="*70)
print("ANOMALY TIMELINE ANALYSIS")
print("="*70)

labels_array = labels.values

# Find anomaly regions
anomaly_indices = np.where(labels_array == 1)[0]
if len(anomaly_indices) > 0:
    # Find contiguous anomaly regions
    anomaly_regions = []
    region_start = anomaly_indices[0]
    region_end = anomaly_indices[0]
    
    for idx in anomaly_indices[1:]:
        if idx == region_end + 1:
            region_end = idx
        else:
            anomaly_regions.append((region_start, region_end))
            region_start = idx
            region_end = idx
    anomaly_regions.append((region_start, region_end))
    
    print(f"\n📊 Found {len(anomaly_regions)} anomaly region(s):")
    print(f"")
    
    total_timesteps = len(labels_array)
    
    for i, (start, end) in enumerate(anomaly_regions):
        duration = end - start + 1
        start_time = format_time(start)
        end_time = format_time(end)
        duration_time = format_time(duration)
        
        print(f"  Region {i+1}:")
        print(f"    Start: timestep {start:,} ({start_time} from beginning)")
        print(f"    End:   timestep {end:,} ({end_time} from beginning)")
        print(f"    Duration: {duration:,} timesteps ({duration_time})")
        print(f"")
        
        # Check if we have enough lead time for prediction
        available_lead_time = start - WINDOW_SIZE
        print(f"    Available data before anomaly: {available_lead_time:,} timesteps ({format_time(available_lead_time)})")
        print(f"    Required prediction horizon: {PREDICTION_HORIZON:,} timesteps ({format_time(PREDICTION_HORIZON)})")
        
        if available_lead_time >= PREDICTION_HORIZON:
            print(f"    ✅ SUFFICIENT data for {format_time(PREDICTION_HORIZON)} prediction")
        else:
            shortfall = PREDICTION_HORIZON - available_lead_time
            print(f"    ⚠️  INSUFFICIENT data - need {format_time(shortfall)} more")
        print(f"")
    
    # Visualize anomaly timeline
    fig, ax = plt.subplots(figsize=(15, 4))
    
    time_minutes = np.arange(len(labels_array)) / SAMPLE_RATE_HZ / 60
    
    ax.fill_between(time_minutes, 0, labels_array, alpha=0.5, color='red', label='Anomaly')
    ax.set_xlabel('Time (minutes)', fontsize=12)
    ax.set_ylabel('Anomaly', fontsize=12)
    ax.set_title('Anomaly Timeline', fontsize=14, fontweight='bold')
    ax.set_ylim(-0.1, 1.1)
    
    # Add prediction horizon marker
    horizon_minutes = PREDICTION_HORIZON / SAMPLE_RATE_HZ / 60
    ax.axvline(x=horizon_minutes, color='blue', linestyle='--', linewidth=2, 
               label=f'Prediction horizon ({format_time(PREDICTION_HORIZON)})')
    
    ax.legend()
    plt.tight_layout()
    plt.show()
    
else:
    print("⚠️  No anomalies found in the dataset!")

## 8. 🔮 Create Predictive Sequences

### Key Concept: True Prediction vs Detection

```
Timeline:  [----OBSERVE----][----GAP----][----PREDICT----]
           |    5 sec      |            |    horizon     |
           
The model observes historical data and predicts if anomalies will occur
in the FUTURE window (after a gap equal to the prediction horizon).
```

In [ ]:
def create_predictive_sequences(data, labels, window_size, prediction_horizon, stride, sample_rate, min_lead_seconds=0):
    """
    Create sequences for PREDICTIVE modeling with CORRECT time calculations.
    
    Args:
        data: Sensor data (normalized)
        labels: Anomaly labels
        window_size: Historical window to observe (timesteps)
        prediction_horizon: How far ahead to predict (timesteps)
        stride: Sliding window stride
        sample_rate: Sampling rate in Hz
    
    Returns:
        X: Historical sequences (input)
        y: Future anomaly labels (target)
        lead_times: Actual lead time for each prediction (timesteps)
        metadata: Additional info about each sample
    """
    print(f"\n{'='*70}")
    print("CREATING PREDICTIVE SEQUENCES")
    print(f"{'='*70}")
    print(f"Historical window: {window_size} timesteps ({format_time(window_size)})")
    print(f"Prediction horizon: {prediction_horizon} timesteps ({format_time(prediction_horizon)})")
    print(f"Stride: {stride} timesteps ({format_time(stride)})")
    min_lead_ts = int(min_lead_seconds * sample_rate)
    print(f"\n🔮 Label strategy: Predict anomalies {format_time(prediction_horizon)} into the future")
    print(f"⏳ Minimum required lead time: {min_lead_seconds}s ({min_lead_ts} timesteps)")
    
    data_array = data.values
    labels_array = labels.values
    
    X_sequences = []
    y_sequences = []
    lead_times = []
    metadata = []
    
    # Need enough data for: current window + prediction horizon
    max_idx = len(data_array) - window_size - prediction_horizon
    
    for i in range(0, max_idx, stride):
        # Historical window (what we observe)
        window = data_array[i:i+window_size]
        
        # Future window (what we want to predict)
        future_start = i + window_size
        future_end = future_start + prediction_horizon
        future_labels = labels_array[future_start:future_end]
        
        # ---------------------------------------------------------------------
        # Labeling with minimum-lead-time constraint:
        # - If min_lead_seconds == 0: any anomaly anywhere in the horizon counts.
        # - If min_lead_seconds > 0: only anomalies that occur at/after min_lead
        #   count as positives. This prevents "predictions" that are effectively
        #   detections (lead_time ~= 0).
        # ---------------------------------------------------------------------
        search_start = min_lead_ts if min_lead_ts > 0 else 0
        eligible_future = future_labels[search_start:]
        eligible_idxs = np.where(eligible_future == 1)[0]
        
        # Label: Will there be an anomaly in the eligible FUTURE window?
        future_anomaly = 1 if eligible_idxs.size > 0 else 0
        
        # Lead time (timesteps from window end to first eligible anomaly)
        if future_anomaly == 1:
            lead_time = int(search_start + eligible_idxs[0])
        else:
            lead_time = prediction_horizon  # No anomaly in eligible horizon
        
        X_sequences.append(window)
        y_sequences.append(future_anomaly)
        lead_times.append(lead_time)
        metadata.append({
            'window_start': i,
            'window_end': i + window_size,
            'future_start': future_start,
            'future_end': future_end
        })
    
    X = np.array(X_sequences)
    y = np.array(y_sequences)
    lead_times = np.array(lead_times)
    
    print(f"\n✅ Created {len(X):,} predictive sequences")
    print(f"X shape: {X.shape} (n_sequences, window_size, n_sensors)")
    print(f"y shape: {y.shape}")
    
    print(f"\n{'='*70}")
    print("PREDICTIVE LABEL DISTRIBUTION")
    print(f"{'='*70}")
    print(f"No future anomaly (0): {(y == 0).sum():,} ({(y == 0).sum() / len(y) * 100:.2f}%)")
    print(f"Future anomaly (1):   {(y == 1).sum():,} ({(y == 1).sum() / len(y) * 100:.2f}%)")
    
    # Analyze lead times for predictions
    if (y == 1).sum() > 0:
        print(f"\n{'='*70}")
        print("EARLY WARNING STATISTICS (CORRECTED UNITS)")
        print(f"{'='*70}")
        anomaly_lead_times = lead_times[y == 1]
        
        avg_lead = anomaly_lead_times.mean()
        median_lead = np.median(anomaly_lead_times)
        min_lead = anomaly_lead_times.min()
        max_lead = anomaly_lead_times.max()
        
        print(f"Average lead time: {avg_lead:.1f} timesteps ({format_time(avg_lead)})")
        print(f"Median lead time:  {median_lead:.1f} timesteps ({format_time(median_lead)})")
        print(f"Min lead time:     {min_lead:.0f} timesteps ({format_time(min_lead)})")
        print(f"Max lead time:     {max_lead:.0f} timesteps ({format_time(max_lead)})")
        
        # Lead time distribution
        print(f"\n📊 Lead Time Distribution:")
        for threshold in [0, 10, 30, 60, 180, 300]:
            threshold_ts = threshold * sample_rate
            if threshold_ts <= prediction_horizon:
                pct = (anomaly_lead_times >= threshold_ts).sum() / len(anomaly_lead_times) * 100
                print(f"   ≥{threshold:3d}s ({threshold_ts:5d} ts): {pct:5.1f}% of predictions")
        
        if min_lead_ts > 0:
            pct_sla = (anomaly_lead_times >= min_lead_ts).sum() / len(anomaly_lead_times) * 100
            print(f"\n✅ SLA coverage: {pct_sla:5.1f}% of positive labels have ≥{min_lead_seconds}s lead time")
        
        # Warning about median
        if median_lead < prediction_horizon * 0.1:
            print(f"\n⚠️  WARNING: Median lead time is very low ({format_time(median_lead)})")
            print(f"   This suggests most 'predictions' have minimal advance warning.")
            print(f"   The model may be detecting rather than truly predicting.")
    
    return X, y, lead_times, metadata

In [ ]:
# Create predictive sequences
X, y, lead_times, metadata = create_predictive_sequences(
    normalized, labels, WINDOW_SIZE, PREDICTION_HORIZON, STRIDE, SAMPLE_RATE_HZ, MIN_LEAD_SECONDS
)

## 9. Visualize Lead Time Distribution

In [ ]:
if (y == 1).sum() > 0:
    anomaly_lead_times = lead_times[y == 1]
    
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # Histogram of lead times
    lead_times_seconds = anomaly_lead_times / SAMPLE_RATE_HZ
    axes[0].hist(lead_times_seconds, bins=50, color='blue', alpha=0.7, edgecolor='black')
    axes[0].axvline(x=np.median(lead_times_seconds), color='red', linestyle='--', 
                    linewidth=2, label=f'Median: {np.median(lead_times_seconds):.1f}s')
    axes[0].axvline(x=np.mean(lead_times_seconds), color='orange', linestyle='--', 
                    linewidth=2, label=f'Mean: {np.mean(lead_times_seconds):.1f}s')
    axes[0].set_xlabel('Lead Time (seconds)', fontsize=12)
    axes[0].set_ylabel('Count', fontsize=12)
    axes[0].set_title('Lead Time Distribution for Anomaly Predictions', fontsize=14, fontweight='bold')
    axes[0].legend()
    
    # Cumulative distribution
    sorted_lead = np.sort(lead_times_seconds)
    cumulative = np.arange(1, len(sorted_lead) + 1) / len(sorted_lead)
    axes[1].plot(sorted_lead, cumulative, linewidth=2, color='blue')
    axes[1].axhline(y=0.5, color='red', linestyle='--', alpha=0.7, label='50% (median)')
    axes[1].set_xlabel('Lead Time (seconds)', fontsize=12)
    axes[1].set_ylabel('Cumulative Proportion', fontsize=12)
    axes[1].set_title('Cumulative Distribution of Lead Times', fontsize=14, fontweight='bold')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Summary statistics
    print(f"\n📊 Lead Time Summary:")
    print(f"   Total anomaly predictions: {len(anomaly_lead_times)}")
    print(f"   Predictions with 0s lead time: {(anomaly_lead_times == 0).sum()} ({(anomaly_lead_times == 0).sum()/len(anomaly_lead_times)*100:.1f}%)")
    print(f"   Predictions with >1min lead time: {(lead_times_seconds >= 60).sum()} ({(lead_times_seconds >= 60).sum()/len(anomaly_lead_times)*100:.1f}%)")

## 10. Train/Val/Test Split

Using **temporal split** for more realistic evaluation.

In [ ]:
print("\n" + "="*70)
print("SPLITTING DATA (Temporal Split)")
print("="*70)

# Use temporal split instead of random split for time series
n_samples = len(X)
train_end = int(n_samples * 0.7)
val_end = int(n_samples * 0.85)

X_train = X[:train_end]
y_train = y[:train_end]
lt_train = lead_times[:train_end]

X_val = X[train_end:val_end]
y_val = y[train_end:val_end]
lt_val = lead_times[train_end:val_end]

X_test = X[val_end:]
y_test = y[val_end:]
lt_test = lead_times[val_end:]

print(f"Train set: {X_train.shape[0]:,} samples (70.0%)")
print(f"Val set:   {X_val.shape[0]:,} samples (15.0%)")
print(f"Test set:  {X_test.shape[0]:,} samples (15.0%)")

print(f"\nClass distribution:")
print(f"  Train - Normal: {(y_train == 0).sum():,}, Anomaly: {(y_train == 1).sum():,}")
print(f"  Val   - Normal: {(y_val == 0).sum():,}, Anomaly: {(y_val == 1).sum():,}")
print(f"  Test  - Normal: {(y_test == 0).sum():,}, Anomaly: {(y_test == 1).sum():,}")

# Check if test set has anomalies
if (y_test == 1).sum() == 0:
    print(f"\n⚠️  WARNING: Test set has NO anomalies!")
    print(f"   This may happen with temporal split if anomalies are early in the data.")
    print(f"   Consider using stratified random split for evaluation.")

In [ ]:
# Alternative: Stratified random split (uncomment if temporal split has no test anomalies)
USE_STRATIFIED_SPLIT = (y_test == 1).sum() == 0

if USE_STRATIFIED_SPLIT:
    print("\n" + "="*70)
    print("USING STRATIFIED SPLIT (fallback)")
    print("="*70)
    
    # First split: 70% train, 30% temp
    X_train, X_temp, y_train, y_temp, lt_train, lt_temp = train_test_split(
        X, y, lead_times, test_size=0.3, random_state=42, stratify=y
    )
    
    # Second split: 15% val, 15% test
    X_val, X_test, y_val, y_test, lt_val, lt_test = train_test_split(
        X_temp, y_temp, lt_temp, test_size=0.5, random_state=42, stratify=y_temp
    )
    
    print(f"Train set: {X_train.shape[0]:,} samples")
    print(f"Val set:   {X_val.shape[0]:,} samples")
    print(f"Test set:  {X_test.shape[0]:,} samples")
    print(f"\nClass distribution (stratified):")
    print(f"  Train - Normal: {(y_train == 0).sum():,}, Anomaly: {(y_train == 1).sum():,}")
    print(f"  Val   - Normal: {(y_val == 0).sum():,}, Anomaly: {(y_val == 1).sum():,}")
    print(f"  Test  - Normal: {(y_test == 0).sum():,}, Anomaly: {(y_test == 1).sum():,}")

## 11. Build Predictive CNN Model

In [ ]:
def build_predictive_cnn(input_shape, n_filters=[64, 128, 256], kernel_sizes=[5, 3, 3], dropout_rate=0.3):
    """
    Build 1D CNN for PREDICTIVE anomaly detection
    """
    print("\nBuilding Predictive 1D CNN model...")
    print(f"Input shape: {input_shape}")
    
    model = models.Sequential(name='Predictive_1D_CNN')
    
    model.add(layers.Input(shape=input_shape))
    
    # Conv Block 1
    model.add(layers.Conv1D(n_filters[0], kernel_sizes[0], padding='same', activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling1D(pool_size=2))
    model.add(layers.Dropout(dropout_rate))
    
    # Conv Block 2
    model.add(layers.Conv1D(n_filters[1], kernel_sizes[1], padding='same', activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling1D(pool_size=2))
    model.add(layers.Dropout(dropout_rate))
    
    # Conv Block 3
    model.add(layers.Conv1D(n_filters[2], kernel_sizes[2], padding='same', activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.GlobalMaxPooling1D())
    model.add(layers.Dropout(dropout_rate))
    
    # Dense layers
    model.add(layers.Dense(128, activation='relu'))
    model.add(layers.Dropout(dropout_rate))
    model.add(layers.Dense(64, activation='relu'))
    model.add(layers.Dropout(dropout_rate))
    
    # Output
    model.add(layers.Dense(1, activation='sigmoid', name='risk_score'))
    
    return model

In [ ]:
# Build model
input_shape = (WINDOW_SIZE, X.shape[2])
model = build_predictive_cnn(input_shape)

# Display architecture
model.summary()

print(f"\n✅ Model built successfully!")
print(f"Total parameters: {model.count_params():,}")

## 12. Compile and Train Model

In [ ]:
# Calculate class weights
n_normal = (y_train == 0).sum()
n_anomaly = (y_train == 1).sum()
class_weight = {0: 1.0, 1: n_normal / n_anomaly if n_anomaly > 0 else 1.0}

print(f"Class weights: {class_weight}")

# Compile model
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        keras.metrics.Precision(name='precision'),
        keras.metrics.Recall(name='recall'),
        keras.metrics.AUC(name='auc')
    ]
)

print("✅ Model compiled successfully!")

In [ ]:
# Setup callbacks
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        os.path.join(MODEL_DIR, f'best_predictive_model_{PREDICTION_MODE}.keras'),
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    )
]

print("✅ Callbacks configured")

In [ ]:
print("\n" + "="*70)
print(f"TRAINING PREDICTIVE MODEL ({PREDICTION_MODE} horizon)")
print("="*70)
print(f"Prediction horizon: {format_time(PREDICTION_HORIZON)}")
print(f"Starting training at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weight,
    callbacks=callbacks,
    verbose=1
)

print(f"\n✅ Training completed at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 13. Training History Visualization

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Loss
axes[0, 0].plot(history.history['loss'], label='Train Loss', linewidth=2)
axes[0, 0].plot(history.history['val_loss'], label='Val Loss', linewidth=2)
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].set_title('Model Loss', fontsize=14, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Accuracy
axes[0, 1].plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
axes[0, 1].plot(history.history['val_accuracy'], label='Val Accuracy', linewidth=2)
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Accuracy')
axes[0, 1].set_title('Model Accuracy', fontsize=14, fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Precision & Recall
axes[1, 0].plot(history.history['precision'], label='Train Precision', linewidth=2)
axes[1, 0].plot(history.history['val_precision'], label='Val Precision', linewidth=2)
axes[1, 0].plot(history.history['recall'], label='Train Recall', linewidth=2, linestyle='--')
axes[1, 0].plot(history.history['val_recall'], label='Val Recall', linewidth=2, linestyle='--')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Score')
axes[1, 0].set_title('Precision & Recall', fontsize=14, fontweight='bold')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# AUC
axes[1, 1].plot(history.history['auc'], label='Train AUC', linewidth=2)
axes[1, 1].plot(history.history['val_auc'], label='Val AUC', linewidth=2)
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('AUC')
axes[1, 1].set_title('ROC AUC', fontsize=14, fontweight='bold')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle(f'Training History - {PREDICTION_MODE} Prediction ({format_time(PREDICTION_HORIZON)})', 
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, f'training_history_{PREDICTION_MODE}.png'), dpi=150, bbox_inches='tight')
plt.show()

## 14. Test Set Evaluation

In [ ]:
print("\n" + "="*70)
print(f"TEST SET EVALUATION ({PREDICTION_MODE} horizon)")
print("="*70)

# Get predictions
risk_scores = model.predict(X_test, verbose=0).flatten()
y_pred = (risk_scores > 0.5).astype(int)

# Classification report
print(f"\nClassification Report (threshold=0.5):")
print(classification_report(y_test, y_pred, target_names=['No Future Anomaly', 'Future Anomaly']))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
print(f"\nConfusion Matrix:")
print(f"                    Predicted")
print(f"           No Anomaly  Future Anomaly")
print(f"Actual No Anomaly       {cm[0,0]:<4}        {cm[0,1]:<4}")
print(f"       Future Anomaly   {cm[1,0]:<4}        {cm[1,1]:<4}")

# ROC-AUC
if len(np.unique(y_test)) > 1:
    roc_auc = roc_auc_score(y_test, risk_scores)
    print(f"\n🎯 ROC-AUC Score: {roc_auc:.4f}")
else:
    print(f"\n⚠️  Cannot calculate ROC-AUC (only one class in test set)")

In [ ]:
# Early warning capability analysis
print("\n" + "="*70)
print("EARLY WARNING CAPABILITY ANALYSIS")
print("="*70)

if (y_test == 1).sum() > 0:
    correct_predictions = (y_pred == 1) & (y_test == 1)
    
    if correct_predictions.sum() > 0:
        warning_lead_times = lt_test[correct_predictions]
        
        print(f"✅ Successfully predicted anomalies: {correct_predictions.sum()}/{(y_test == 1).sum()}")
        print(f"\nLead time statistics (CORRECT UNITS):")
        print(f"  Average: {format_time(warning_lead_times.mean())}")
        print(f"  Median:  {format_time(np.median(warning_lead_times))}")
        print(f"  Min:     {format_time(warning_lead_times.min())}")
        print(f"  Max:     {format_time(warning_lead_times.max())}")
        
        # Practical warning percentages
        print(f"\n📊 Practical Early Warning:")
        for threshold_sec in [0, 10, 30, 60, 120, 180]:
            threshold_ts = threshold_sec * SAMPLE_RATE_HZ
            pct = (warning_lead_times >= threshold_ts).sum() / len(warning_lead_times) * 100
            print(f"   ≥{threshold_sec:3d} seconds: {pct:5.1f}% of correct predictions")

        if MIN_LEAD_SECONDS > 0:
            pct_sla = (warning_lead_times >= MIN_LEAD_SECONDS * SAMPLE_RATE_HZ).sum() / len(warning_lead_times) * 100
            print(f"\n✅ SLA met (lead ≥ {MIN_LEAD_SECONDS}s): {pct_sla:5.1f}% of correct predictions")
    else:
        print("⚠️  No correct anomaly predictions")
else:
    print("⚠️  No anomalies in test set")

## 15. ROC and PR Curves

In [ ]:
if len(np.unique(y_test)) > 1:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # ROC Curve
    fpr, tpr, thresholds = roc_curve(y_test, risk_scores)
    roc_auc = roc_auc_score(y_test, risk_scores)
    
    axes[0].plot(fpr, tpr, linewidth=2, label=f'ROC (AUC = {roc_auc:.4f})')
    axes[0].plot([0, 1], [0, 1], 'k--', linewidth=1)
    axes[0].set_xlabel('False Positive Rate', fontsize=12)
    axes[0].set_ylabel('True Positive Rate', fontsize=12)
    axes[0].set_title('ROC Curve', fontsize=14, fontweight='bold')
    axes[0].legend(loc='lower right')
    axes[0].grid(True, alpha=0.3)
    
    # PR Curve
    precision, recall, thresholds = precision_recall_curve(y_test, risk_scores)
    
    axes[1].plot(recall, precision, linewidth=2)
    axes[1].set_xlabel('Recall', fontsize=12)
    axes[1].set_ylabel('Precision', fontsize=12)
    axes[1].set_title('Precision-Recall Curve', fontsize=14, fontweight='bold')
    axes[1].grid(True, alpha=0.3)
    
    plt.suptitle(f'{PREDICTION_MODE} Prediction ({format_time(PREDICTION_HORIZON)})', 
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(MODEL_DIR, f'evaluation_{PREDICTION_MODE}.png'), dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("⚠️  Cannot plot curves (only one class in test set)")

## 16. Save Model and Configuration

In [ ]:
# Save model
model_path = os.path.join(MODEL_DIR, f'predictive_cnn_model_{PREDICTION_MODE}.keras')
model.save(model_path)
print(f"✅ Model saved to: {model_path}")

# Save configuration
config = {
    'prediction_mode': PREDICTION_MODE,
    'sample_rate_hz': SAMPLE_RATE_HZ,
    'window_size_timesteps': WINDOW_SIZE,
    'window_size_seconds': WINDOW_SIZE / SAMPLE_RATE_HZ,
    'prediction_horizon_timesteps': PREDICTION_HORIZON,
    'prediction_horizon_seconds': PREDICTION_HORIZON / SAMPLE_RATE_HZ,
    'prediction_horizon_minutes': PREDICTION_HORIZON / SAMPLE_RATE_HZ / 60,
    'stride': STRIDE,
    'n_sensors': X.shape[2],
    'sensor_columns': list(pivot.columns),
    'created_at': datetime.now().isoformat()
}

config_path = os.path.join(MODEL_DIR, f'config_{PREDICTION_MODE}.json')
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)
print(f"✅ Configuration saved to: {config_path}")

# Save scalers
scalers_path = os.path.join(MODEL_DIR, f'scalers_{PREDICTION_MODE}.pkl')
joblib.dump(scalers, scalers_path)
print(f"✅ Scalers saved to: {scalers_path}")

## 17. 📊 Summary & Comparison

### Results for Current Configuration

In [ ]:
print("="*70)
print(f"SUMMARY - {PREDICTION_MODE} PREDICTION MODE")
print("="*70)
print(f"")
print(f"📊 Configuration:")
print(f"   Prediction horizon: {format_time(PREDICTION_HORIZON)}")
print(f"   Historical window:  {format_time(WINDOW_SIZE)}")
print(f"   Sampling rate:      {SAMPLE_RATE_HZ} Hz")
print(f"")

if len(np.unique(y_test)) > 1:
    print(f"📈 Performance Metrics:")
    print(f"   ROC-AUC:   {roc_auc:.4f}")
    print(f"   Accuracy:  {(y_pred == y_test).mean():.4f}")
    
    if (y_test == 1).sum() > 0:
        recall = (y_pred[y_test == 1] == 1).sum() / (y_test == 1).sum()
        print(f"   Recall:    {recall:.4f}")
        
        if (y_pred == 1).sum() > 0:
            precision = (y_test[y_pred == 1] == 1).sum() / (y_pred == 1).sum()
            print(f"   Precision: {precision:.4f}")
    print(f"")
    
    if (y_test == 1).sum() > 0 and correct_predictions.sum() > 0:
        print(f"⏱️  Early Warning Statistics:")
        print(f"   Median lead time: {format_time(np.median(warning_lead_times))}")
        print(f"   Mean lead time:   {format_time(warning_lead_times.mean())}")
        print(f"   Max lead time:    {format_time(warning_lead_times.max())}")
        print(f"")
        
        # Honest assessment
        median_seconds = np.median(warning_lead_times) / SAMPLE_RATE_HZ
        if median_seconds < 10:
            print(f"⚠️  HONEST ASSESSMENT:")
            print(f"   Median lead time of {median_seconds:.1f}s suggests the model is primarily")
            print(f"   DETECTING anomalies rather than truly PREDICTING them.")
            print(f"   Most 'predictions' have minimal advance warning.")
        elif median_seconds < 60:
            print(f"✅ ASSESSMENT:")
            print(f"   Model provides meaningful short-term predictions.")
            print(f"   Suitable for automated immediate response systems.")
        else:
            print(f"🎉 ASSESSMENT:")
            print(f"   Model provides meaningful advance predictions!")
            print(f"   Suitable for proactive maintenance planning.")
else:
    print(f"⚠️  Insufficient test data for full evaluation")

print(f"\n" + "="*70)

## 18. 🔄 Run Multiple Horizons Comparison

To compare different prediction horizons, run this notebook multiple times with different `PREDICTION_MODE` values:

1. Set `PREDICTION_MODE = 'SHORT'` → Run all cells → Note results
2. Set `PREDICTION_MODE = 'MEDIUM'` → Run all cells → Note results  
3. Set `PREDICTION_MODE = 'LONG'` → Run all cells → Note results

### Expected Trade-offs:

| Horizon | Expected Accuracy | Practical Use |
|---------|-------------------|---------------|
| 5 seconds | Highest | Automated shutdown |
| 3 minutes | Medium | Alert operators |
| 5 minutes | Lower | Schedule maintenance |

Longer prediction horizons are inherently more difficult because:
- Patterns that precede anomalies may not be visible far in advance
- More intervening factors can affect outcomes
- Signal-to-noise ratio decreases

In [ ]:
print("\n" + "="*70)
print("NEXT STEPS")
print("="*70)
print(f"")
print(f"To test other prediction horizons:")
print(f"")
print(f"1. Change PREDICTION_MODE at the top of Section 2:")
print(f"   - 'SHORT'  → 5 seconds prediction")
print(f"   - 'MEDIUM' → 3 minutes prediction")
print(f"   - 'LONG'   → 5 minutes prediction")
print(f"   - 'CUSTOM' → Set CUSTOM_HORIZON_SECONDS")
print(f"")
print(f"2. Run all cells again (Kernel → Restart & Run All)")
print(f"")
print(f"3. Compare results across different horizons")
print(f"")
print(f"Current mode: {PREDICTION_MODE} ({format_time(PREDICTION_HORIZON)})")
print("="*70)